In [4]:

import pandas as pd
from pathlib import Path

# ajusta o caminho se o notebook não estiver na raiz do projeto
df = pd.read_csv("../data/raw/dataset_geocodificado_raw.csv")

df.head()

,titulo,preco_raw,preco_eur,morada,tipologia,area_raw,area_m2,piso,link,freguesia,lat,lon,dist_centro_km,dist_universidade_km,dist_polo2_km,dist_chuc_km
0,Apartamento T3 - Olivais - Coimbra/Venda,279 000 €,279000.0,"Quinta da Maia, Santo António dos Olivais, Coi...",T3,92 m²,92.0,2 andar,/pt/anuncio/apartamento-t3-olivais-coimbra-ven...,santo_antonio_olivais,40.211722,-8.401422,2.357213,2.025556,2.100024,2.445530
1,Duplex à venda na Rua Pêro Vaz de Caminha,435 000 €,435000.0,"Rua Pêro Vaz de Caminha, Bairro Norton de Mato...",T4,106 m²,106.0,4 andar,/pt/anuncio/duplex-a-venda-na-rua-pero-vaz-de-...,santo_antonio_olivais,NaN,NaN,NaN,NaN,NaN,NaN
2,Apartamento T9 em Dúplex com garagem e duas en...,379 000 €,379000.0,"Rua Teófilo Braga, Alto de S. João - Portela -...",T5,100 m²,100.0,NaN,/pt/anuncio/apartamento-t9-em-duplex-com-garag...,santo_antonio_olivais,NaN,NaN,NaN,NaN,NaN,NaN
3,T1 de Luxo com Varanda | Garagem | Olivais Ter...,310 000 €,310000.0,"Calçada de São Sebastião, Olivais, Santo Antón...",T1,70 m²,70.0,NaN,/pt/anuncio/t1-de-luxo-com-varanda-garagem-oli...,santo_antonio_olivais,40.217736,-8.399831,2.598267,2.390149,2.736917,3.035898
4,"Investimento em Coimbra: Prédio com 8,7% Yield...",1 050 000 €,1050000.0,"Largo da Conchada, Celas, Santo António dos Ol...",T9+,781 m²,781.0,NaN,/pt/anuncio/investimento-em-coimbra-predio-com...,santo_antonio_olivais,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 1. Limpeza básica
df = df.drop_duplicates(subset="link")          # anúncios repetidos
df = df.dropna(subset=["preco_eur", "area_m2"]) # sem preço/área não dá para calcular nada

# 2. A métrica central do projeto
df["preco_m2"] = df["preco_eur"] / df["area_m2"]

# 3. Remover outliers óbvios (erros de scraping, terrenos mal classificados, etc.)
df = df[(df["preco_m2"] > 300) & (df["preco_m2"] < 8000)]

# 4. Preço "esperado" por freguesia (a mediana)
mediana_freg = df.groupby("freguesia")["preco_m2"].median()
df["preco_m2_mediana_freguesia"] = df["freguesia"].map(mediana_freg)

# 5. Desconto/prémio face à zona
df["desvio_pct"] = (df["preco_m2"] - df["preco_m2_mediana_freguesia"]) / df["preco_m2_mediana_freguesia"]

# 6. As "oportunidades"
oportunidades = df.sort_values("desvio_pct").head(15)
oportunidades[["titulo", "freguesia", "preco_eur", "area_m2", "preco_m2", "desvio_pct",""]]

,titulo,freguesia,preco_eur,area_m2,preco_m2,desvio_pct
1222,"Prédio para reconstrução, Coimbra, Santa Clara...",santa_clara_castelo_viegas,1400000.0,1962.00,713.557594,-0.840739
856,Vende-se Prédio numa das principais ruas da Ba...,coimbra_centro,500000.0,750.00,666.666667,-0.818182
789,2 prédios com projeto aprovado - junto à via c...,coimbra_centro,400000.0,550.00,727.272727,-0.801653
1218,"Prédio+Terreno , Santa Clara",santa_clara_castelo_viegas,380000.0,400.00,950.000000,-0.787966
1281,Prédio Venda em São Martinho do Bispo e Ribei...,sao_martinho_bispo,280000.0,427.19,655.446054,-0.780297
574,"Apartamento T5 com 241,67m2 | Oficina com 373,...",eiras_sao_paulo_frades,490500.0,669.00,733.183857,-0.778232
1271,Lote de terreno C/projeto aprovado,sao_martinho_bispo,350000.0,500.00,700.000000,-0.765363
906,Conjunto 4 Apartamentos T2 para venda | Castel...,santa_clara_castelo_viegas,295000.0,269.00,1096.654275,-0.755234
16,Prédio para Investimento | Olivais – Coimbra,santo_antonio_olivais,575000.0,500.00,1150.000000,-0.743462
497,️ Prédio no centro de Coimbra — junto ao Hospi...,santo_antonio_olivais,800000.0,670.00,1194.029851,-0.733639
